# BaddieVision — Colab setup

This notebook clones BaddieVision with its submodules, mounts Google Drive, verifies and extracts the selected data packages, installs dependencies, and checks the runtime.

Before running it, upload the desired ZIP files to **My Drive/BaddieVision/**.

## 1. Select a GPU runtime

In Colab, choose **Runtime → Change runtime type → T4 GPU** (or another available GPU), then run the cells from top to bottom.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 2. Choose packages

- **features**: enough for shot-classifier and in-play/LSTM training.
- **models**: needed for inference and for regenerating shuttle features.
- **raw**: needed when regenerating features from the included videos; it takes about 1.1 GB.

Change the booleans below before running the setup.

In [ ]:
from pathlib import Path

GITHUB_URL = "https://github.com/MaxLinCode/BaddieVision.git"
REPO_DIR = Path("/content/BaddieVision")
DRIVE_DIR = Path("/content/drive/MyDrive/BaddieVision")

# Pick only what this Colab session needs.
USE_RAW = False
USE_FEATURES = True
USE_MODELS = True

PACKAGES = {
    "raw": {
        "enabled": USE_RAW,
        "filename": "badminton-raw-v1-2026-07-01.zip",
        "sha256": "20fa5fb312e0a3d8e6addaa15f4350ec4349aadadf169f95c117a95a46338da0",
    },
    "features": {
        "enabled": USE_FEATURES,
        "filename": "badminton-features-v1-2026-07-01.zip",
        "sha256": "df5c0977014de67b54f93301afbb08ee63480f5a14e6ed188f71762f8a9e6a38",
    },
    "models": {
        "enabled": USE_MODELS,
        "filename": "badminton-models-v1-2026-07-01.zip",
        "sha256": "0f4db8f05a78d05e879356ee9fedb45aeca0e8fa1e16a94ca8226c3af75987d3",
    },
}

print(f"Drive package folder: {DRIVE_DIR}")
print("Selected:", [name for name, item in PACKAGES.items() if item["enabled"]])

In [ ]:
import subprocess

if (REPO_DIR / ".git").is_dir():
    print("Repository already exists; updating it.")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "submodule", "update", "--init", "--recursive"],
        check=True,
    )
else:
    subprocess.run(
        ["git", "clone", "--recurse-submodules", GITHUB_URL, str(REPO_DIR)],
        check=True,
    )

print(f"Repository ready at {REPO_DIR}")

In [ ]:
import hashlib
import shutil
import zipfile


def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        while chunk := stream.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


selected = [item for item in PACKAGES.values() if item["enabled"]]
missing = [DRIVE_DIR / item["filename"] for item in selected if not (DRIVE_DIR / item["filename"]).is_file()]
if missing:
    available = sorted(path.name for path in DRIVE_DIR.glob("*.zip")) if DRIVE_DIR.exists() else []
    raise FileNotFoundError(
        "Missing selected package(s):\n  "
        + "\n  ".join(str(path) for path in missing)
        + f"\nZIPs currently in Drive folder: {available}"
    )

required_uncompressed = 0
for item in selected:
    archive = DRIVE_DIR / item["filename"]
    with zipfile.ZipFile(archive) as zf:
        unsafe = [name for name in zf.namelist() if Path(name).is_absolute() or ".." in Path(name).parts]
        if unsafe:
            raise ValueError(f"Unsafe archive paths in {archive.name}: {unsafe[:5]}")
        required_uncompressed += sum(info.file_size for info in zf.infolist())

free_space = shutil.disk_usage("/content").free
print(f"Approximate extracted size: {required_uncompressed / 1024**3:.2f} GiB")
print(f"Free Colab disk space:      {free_space / 1024**3:.2f} GiB")
if required_uncompressed > free_space:
    raise RuntimeError("Not enough free Colab disk space for the selected packages.")

for item in selected:
    archive = DRIVE_DIR / item["filename"]
    print(f"\nVerifying {archive.name} ...")
    actual_hash = sha256_file(archive)
    if actual_hash != item["sha256"]:
        raise ValueError(
            f"Checksum mismatch for {archive.name}\n"
            f"Expected: {item['sha256']}\nActual:   {actual_hash}"
        )
    print(f"Extracting into {REPO_DIR} ...")
    subprocess.run(["unzip", "-q", "-o", str(archive), "-d", str(REPO_DIR)], check=True)
    print(f"Ready: {archive.name}")

## 3. Install dependencies

Colab already provides PyTorch. The project requirements will install the remaining runtime dependencies.

In [ ]:
%cd /content/BaddieVision
%pip install -q -r requirements.txt

In [ ]:
import os
import sys
import torch

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Working directory:", Path.cwd())
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

checks = {
    "shot features": REPO_DIR / "clip_features",
    "in-play arrays": REPO_DIR / "InPlay/data",
    "models": REPO_DIR / "models",
    "raw videos": REPO_DIR / "videos",
}
for label, path in checks.items():
    count = sum(1 for item in path.rglob("*") if item.is_file()) if path.exists() else 0
    print(f"{label:16} {str(path.exists()):5}  ({count} files)")

## 4. Run a workflow

Run only the cell for the workflow you want. Training cells are disabled by default to prevent an accidental long run.

In [ ]:
# Requires USE_FEATURES = True
RUN_SHOT_CLASSIFIER_TRAINING = False

if RUN_SHOT_CLASSIFIER_TRAINING:
    subprocess.run([sys.executable, "src/train_shot_classifier.py"], cwd=REPO_DIR, check=True)
else:
    print("Set RUN_SHOT_CLASSIFIER_TRAINING = True to start training.")

In [ ]:
# Requires USE_FEATURES = True
RUN_INPLAY_TRAINING = False

if RUN_INPLAY_TRAINING:
    subprocess.run([sys.executable, "InPlay/train_lstm_model.py"], cwd=REPO_DIR, check=True)
else:
    print("Set RUN_INPLAY_TRAINING = True to start training.")

In [ ]:
# Requires USE_RAW = True and USE_MODELS = True.
# This can be a long-running GPU workflow.
RUN_FEATURE_EXTRACTION = False

if RUN_FEATURE_EXTRACTION:
    subprocess.run([sys.executable, "src/extract_features.py"], cwd=REPO_DIR, check=True)
    subprocess.run([sys.executable, "src/extract_clip_features.py"], cwd=REPO_DIR, check=True)
else:
    print("Set RUN_FEATURE_EXTRACTION = True to regenerate features.")

## 5. Save results before the runtime ends

`/content` is temporary. Copy any trained models or outputs back to Drive.

In [ ]:
from datetime import datetime

SAVE_RESULTS = False

if SAVE_RESULTS:
    destination = DRIVE_DIR / "colab-results" / datetime.now().strftime("%Y%m%d-%H%M%S")
    destination.mkdir(parents=True, exist_ok=True)
    for relative in ["models", "InPlay/models", "outputs", "InPlay/outputs"]:
        source = REPO_DIR / relative
        if source.exists():
            shutil.copytree(source, destination / relative, dirs_exist_ok=True)
    print("Saved results to:", destination)
else:
    print("Set SAVE_RESULTS = True when you are ready to copy results to Drive.")